# Task 4: Model evaluation and analysis

This section implements only evaluation, test prediction, and submission creation.


In [ ]:
from pathlib import Path
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import confusion_matrix


In [ ]:
PLOTS_DIR = Path("assets") / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png"}
RANDOM_SEED = 42


In [ ]:
def get_class_names(label_map: dict) -> list[str]:
    """Return class names ordered by numeric label index."""

    # We support two possible mapping styles from earlier tasks:
    # 1) {0: "AnnualCrop", 1: "Forest", ...}
    # 2) {"AnnualCrop": 0, "Forest": 1, ...}
    if not label_map:
        raise ValueError("label_map is empty.")

    first_key = next(iter(label_map.keys()))

    # If keys are integers, map is already index -> class.
    if isinstance(first_key, int):
        return [label_map[i] for i in sorted(label_map.keys())]

    # If keys are class names, invert to index -> class.
    inverse = {idx: name for name, idx in label_map.items()}
    return [inverse[i] for i in sorted(inverse.keys())]


In [ ]:
def plot_training_history(history: dict) -> None:
    """Plot training/validation loss and validation accuracy over epochs."""

    # Make sure required metrics exist.
    needed = {"train_loss", "val_loss", "val_accuracy"}
    if not needed.issubset(history.keys()):
        raise ValueError(f"history must contain keys: {needed}")

    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: train and validation loss. Lower is better.
    axes[0].plot(epochs, history["train_loss"], label="Train Loss")
    axes[0].plot(epochs, history["val_loss"], label="Validation Loss")
    axes[0].set_title("Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Right: validation accuracy. Higher is better.
    axes[1].plot(epochs, history["val_accuracy"], label="Validation Accuracy", color="green")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "training_history.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
def collect_predictions(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Collect true labels, predicted labels, and max probabilities from a dataloader."""

    # eval() disables dropout randomness and uses batchnorm eval behavior.
    model.eval()

    all_true: list[int] = []
    all_pred: list[int] = []
    all_prob: list[float] = []

    # no_grad() saves memory and speeds up inference.
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            # softmax converts logits to probability-like values in [0,1].
            probs = torch.softmax(logits, dim=1)
            max_probs = probs.max(dim=1).values

            all_true.extend(labels.cpu().numpy().tolist())
            all_pred.extend(preds.cpu().numpy().tolist())
            all_prob.extend(max_probs.cpu().numpy().tolist())

    return np.array(all_true), np.array(all_pred), np.array(all_prob)


In [ ]:
def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str]
) -> None:
    """Plot and save confusion matrix using matplotlib only."""

    # Confusion matrix shows where classes are predicted correctly or confused.
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix (Validation Set)")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.colorbar()

    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right")
    plt.yticks(ticks, class_names)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
def plot_misclassified_samples(
    model: nn.Module,
    df: pd.DataFrame,
    class_names: list[str],
    num_samples: int = 5,
    device: torch.device | None = None
) -> None:
    """Plot up to num_samples misclassified images from the validation dataframe."""

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Use the same deterministic validation transform used during evaluation.
    _, val_transform = get_transforms()

    model.eval()
    wrong_rows: list[tuple[pd.Series, int]] = []

    with torch.no_grad():
        for _, row in df.iterrows():
            img_path = Path(row["folder"]) / row["file_name"]
            image = Image.open(img_path).convert("RGB")
            x = val_transform(image).unsqueeze(0).to(device)

            logits = model(x)
            pred = int(torch.argmax(logits, dim=1).item())
            true = int(row["label"])

            if pred != true:
                wrong_rows.append((row, pred))

    if len(wrong_rows) == 0:
        print("No misclassified samples found.")
        return

    random.seed(RANDOM_SEED)
    selected = random.sample(wrong_rows, k=min(num_samples, len(wrong_rows)))

    cols = len(selected)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
    if cols == 1:
        axes = [axes]

    for ax, (row, pred) in zip(axes, selected):
        image = Image.open(Path(row["folder"]) / row["file_name"]).convert("RGB")
        true = int(row["label"])
        ax.imshow(image)
        ax.set_title(f"True: {class_names[true]}\nPred: {class_names[pred]}")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "misclassified_samples.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
def _natural_key(file_name: str):
    """Helper key for natural sorting (2.jpg before 10.jpg)."""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r"(\d+)", file_name)]


def preprocess_test(data_folder: str) -> pd.DataFrame:
    """Preprocess test folder that contains images directly (no class subfolders)."""

    test_path = Path(data_folder)

    # Test data layout is different from train data:
    # here all images are directly under one folder.
    if not test_path.exists() or not test_path.is_dir():
        raise FileNotFoundError(f"Test folder not found: {test_path}")

    rows: list[dict[str, str]] = []
    for file_path in sorted(test_path.iterdir(), key=lambda p: _natural_key(p.name)):
        if file_path.name.startswith('.'):
            continue
        if file_path.is_file() and file_path.suffix.lower() in VALID_EXTS:
            rows.append({"folder": str(test_path), "file_name": file_path.name})

    test_df = pd.DataFrame(rows, columns=["folder", "file_name"])

    if test_df.empty:
        raise ValueError("No valid test images found.")

    return test_df


In [ ]:
class SatelliteTestDataset(Dataset):
    """Dataset for unlabeled test images."""

    def __init__(self, df: pd.DataFrame, transform: transforms.Compose | None = None) -> None:
        """Store test dataframe and optional transform."""
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        """Return number of test images."""
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, str]:
        """Load one test image and return image tensor with file name."""
        row = self.df.iloc[idx]
        img_path = Path(row["folder"]) / row["file_name"]
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, row["file_name"]


In [ ]:
def predict_test_set(
    model: nn.Module,
    test_df: pd.DataFrame,
    class_names: list[str],
    batch_size: int = 64,
    device: torch.device | None = None
) -> pd.DataFrame:
    """Predict classes for the test set and return file_name + predicted_class."""

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Reuse validation transform for deterministic test-time preprocessing.
    _, val_transform = get_transforms()

    test_dataset = SatelliteTestDataset(test_df, transform=val_transform)

    # shuffle=False is important to keep output order same as test_df order.
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    rows: list[dict[str, str]] = []

    with torch.no_grad():
        for images, file_names in test_loader:
            images = images.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            for file_name, pred_idx in zip(file_names, preds):
                rows.append({
                    "file_name": file_name,
                    "predicted_class": class_names[int(pred_idx)]
                })

    return pd.DataFrame(rows, columns=["file_name", "predicted_class"])


In [ ]:
def create_submission(
    predictions_df: pd.DataFrame,
    output_path: str = "submission.csv"
) -> None:
    """Save challenge submission file without header and without index."""

    needed_cols = {"file_name", "predicted_class"}
    if not needed_cols.issubset(predictions_df.columns):
        raise ValueError(f"predictions_df must contain columns: {needed_cols}")

    # Challenge format requires no header row, so header=False is mandatory.
    predictions_df[["file_name", "predicted_class"]].to_csv(
        output_path,
        index=False,
        header=False,
    )

    print(f"Submission file saved to: {output_path}")


In [ ]:
# Example usage for Task 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = get_class_names(label_map)

model = SatelliteCNN(num_classes=10)
model.load_state_dict(torch.load("assets/weights/best_model.pt", map_location=device))
model.to(device)
model.eval()

plot_training_history(history)

y_true, y_pred, y_prob = collect_predictions(
    model=model,
    dataloader=val_loader,
    device=device
)

plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    class_names=class_names
)

plot_misclassified_samples(
    model=model,
    df=val_df,
    class_names=class_names,
    num_samples=5,
    device=device
)

test_df = preprocess_test("test_data")

predictions_df = predict_test_set(
    model=model,
    test_df=test_df,
    class_names=class_names,
    batch_size=64,
    device=device
)

create_submission(
    predictions_df=predictions_df,
    output_path="submission.csv"
)

print(predictions_df.head())
